In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Main table from your screenshot
TABLE_NAME = "workspace.default.sephora_reviews_enrichedsn"

df = spark.table(TABLE_NAME)

display(df.limit(10))
print("Rows:", df.count())
print("Columns:", len(df.columns))
print(df.columns)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# SETTINGS
TEST_FRAC = 0.2
SEED = 42
MIN_USER_REVIEWS = 5
MIN_ITEM_REVIEWS = 20
MIN_CO_RATINGS = 5
#MIN_CO_RATINGS = 3
TOP_K_NEIGHBORS = 10
#TOP_K_NEIGHBORS = 30
MIN_SIMILARITY = 0.05
HOLDOUT_TABLE = "workspace.default.sephora_test_holdout"

print("Configured per-user split:", int((1-TEST_FRAC)*100), "/", int(TEST_FRAC*100))
print("Min user reviews:", MIN_USER_REVIEWS)
print("Min item reviews:", MIN_ITEM_REVIEWS)
print("Min co-ratings:", MIN_CO_RATINGS)
print("Top-K neighbors:", TOP_K_NEIGHBORS)


In [0]:
print("Holdout table will be written after the train/test split is created:", HOLDOUT_TABLE)


In [0]:
from pyspark.sql import functions as F

ratings_raw = (
    df.select(
        F.col("author_id").cast("string").alias("user_id"),
        F.col("product_id").cast("string").alias("item_id"),
        F.expr("try_cast(rating as double)").alias("rating"),
        F.col("product_name_info").cast("string").alias("product_name")
    )
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("item_id").isNotNull())
    .filter(F.col("rating").isNotNull())
)

display(ratings_raw.limit(10))
print("Ratings rows:", ratings_raw.count())


In [0]:
# Remove impossible ratings and duplicate user-item pairs
# If a user reviewed the same product multiple times, keep the mean rating
ratings_clean = (
    ratings_raw
    .filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    .groupBy("user_id", "item_id")
    .agg(
        F.avg("rating").alias("rating"),
        F.first("product_name", ignorenulls=True).alias("product_name")
    )
)

display(ratings_clean.limit(10))
print("Clean ratings rows:", ratings_clean.count())


In [0]:
# Minimum activity thresholds are applied BEFORE splitting
active_users = (
    ratings_clean.groupBy("user_id")
    .count()
    .filter(F.col("count") >= MIN_USER_REVIEWS)
    .select("user_id")
)

popular_items = (
    ratings_clean.groupBy("item_id")
    .count()
    .filter(F.col("count") >= MIN_ITEM_REVIEWS)
    .select("item_id")
)

ratings_filtered = (
    ratings_clean.join(active_users, on="user_id", how="inner")
    .join(popular_items, on="item_id", how="inner")
)

# Per-user 80/20 random split so every user keeps history in train
w_split = Window.partitionBy("user_id").orderBy(F.rand(SEED))

ratings_split = (
    ratings_filtered
    .withColumn("row_num", F.row_number().over(w_split))
    .withColumn("user_count", F.count("*").over(Window.partitionBy("user_id")))
    .withColumn(
        "test_cutoff",
        F.when(
            F.col("user_count") >= 2,
            F.greatest(F.lit(1), F.floor(F.col("user_count") * F.lit(TEST_FRAC)).cast("int"))
        ).otherwise(F.lit(0))
    )
    .withColumn("is_test", F.col("row_num") <= F.col("test_cutoff"))
)

ratings_train = (
    ratings_split
    .filter(~F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

ratings_test = (
    ratings_split
    .filter(F.col("is_test"))
    .drop("row_num", "user_count", "test_cutoff", "is_test")
)

# Continue the notebook using TRAIN only
ratings = ratings_train

ratings_test.write.format("delta").mode("overwrite").saveAsTable(HOLDOUT_TABLE)

print("Filtered ratings rows (full eligible set):", ratings_filtered.count())
print("Train rows:", ratings_train.count())
print("Test rows:", ratings_test.count())
print("Train users:", ratings_train.select("user_id").distinct().count())
print("Test users:", ratings_test.select("user_id").distinct().count())
print("Train items:", ratings_train.select("item_id").distinct().count())
print("Test items:", ratings_test.select("item_id").distinct().count())

display(ratings.limit(10))
display(ratings_test.limit(10))


In [0]:
# Mean-center TRAIN ratings by user to reduce user scoring bias
user_avg = ratings.groupBy("user_id").agg(F.avg("rating").alias("user_avg_rating"))

ratings_centered = (
    ratings.join(user_avg, on="user_id", how="left")
    .withColumn("rating_centered", F.col("rating") - F.col("user_avg_rating"))
)

display(ratings_centered.limit(10))


In [0]:
# Self-join on item_id to find user pairs who rated the same item in TRAIN only
# Keep only user_a < user_b to avoid duplicates
user_pairs = (
    ratings_centered.alias("a")
    .join(ratings_centered.alias("b"), on="item_id")
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.col("item_id"),
        F.col("a.user_id").alias("user_a"),
        F.col("b.user_id").alias("user_b"),
        F.col("a.rating_centered").alias("rating_a"),
        F.col("b.rating_centered").alias("rating_b")
    )
)

display(user_pairs.limit(10))
print("User pair rows:", user_pairs.count())


In [0]:
# Cosine similarity on TRAIN centered ratings
user_similarity = (
    user_pairs
    .groupBy("user_a", "user_b")
    .agg(
        F.sum(F.col("rating_a") * F.col("rating_b")).alias("dot_product"),
        F.sqrt(F.sum(F.col("rating_a") * F.col("rating_a"))).alias("norm_a"),
        F.sqrt(F.sum(F.col("rating_b") * F.col("rating_b"))).alias("norm_b"),
        F.count("*").alias("co_rating_count")
    )
    .withColumn(
        "similarity",
        F.when(
            (F.col("norm_a") > 0) & (F.col("norm_b") > 0),
            F.col("dot_product") / (F.col("norm_a") * F.col("norm_b"))
        ).otherwise(F.lit(0.0))
    )
)

# Keep only reliable similarities
user_similarity_filtered = (
    user_similarity
    .filter(F.col("co_rating_count") >= MIN_CO_RATINGS)
    .filter(F.col("similarity") >= MIN_SIMILARITY)
    .filter(F.col("similarity").isNotNull())
)

display(user_similarity_filtered.orderBy(F.desc("similarity")).limit(20))
print("Similarity rows:", user_similarity_filtered.count())


In [0]:
sim_ab = user_similarity_filtered.select(
    F.col("user_a").alias("user_id"),
    F.col("user_b").alias("neighbor_user_id"),
    "similarity",
    "co_rating_count"
)

sim_ba = user_similarity_filtered.select(
    F.col("user_b").alias("user_id"),
    F.col("user_a").alias("neighbor_user_id"),
    "similarity",
    "co_rating_count"
)

user_neighbors = sim_ab.unionByName(sim_ba)

display(user_neighbors.orderBy(F.desc("similarity")).limit(20))


In [0]:
w = Window.partitionBy("user_id").orderBy(F.desc("similarity"), F.desc("co_rating_count"))

user_neighbors_topk = (
    user_neighbors
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= TOP_K_NEIGHBORS)
    .drop("rank")
)

display(user_neighbors_topk.limit(20))
print("Top-K neighbor rows:", user_neighbors_topk.count())


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.default.sephora_user_user_similarity")
user_neighbors_topk.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.sephora_user_user_similarity"
)

print("Saved table: workspace.default.sephora_user_user_similarity")


In [0]:
TARGET_USER_ID = "10004195528"   # replace with any user_id you want

similar_users = (
    spark.table("workspace.default.sephora_user_user_similarity")
         .filter(F.col("user_id") == TARGET_USER_ID)
         .orderBy(F.desc("similarity"), F.desc("co_rating_count"))
)

display(similar_users.limit(20))


In [0]:
# Replace with a real user_id from your TRAIN data
TARGET_USER_ID = "10004195528"
# 1741593524
# 10725439731
ratings_used = ratings.select("user_id", "item_id", "rating")
neighbors = spark.table("workspace.default.sephora_user_user_similarity")

# Ratings already made by the target user in TRAIN
user_history = ratings_used.filter(F.col("user_id") == TARGET_USER_ID)

display(user_history)


In [0]:
# Join similar users to the items they rated
neighbor_history = (
    neighbors.alias("n")
    .join(
        ratings_used.alias("r"),
        F.col("n.neighbor_user_id") == F.col("r.user_id"),
        how="inner"
    )
    .select(
        F.col("n.user_id"),
        F.col("r.item_id").alias("candidate_item_id"),
        F.col("r.rating").alias("neighbor_rating"),
        F.col("n.similarity"),
        F.col("n.co_rating_count")
    )
)

# Product names for display
item_lookup = ratings.select("item_id", "product_name").dropDuplicates(["item_id"])

# Remove items the target user already rated
already_rated = user_history.select(F.col("item_id").alias("candidate_item_id"))

user_recommendations = (
    neighbor_history
    .join(already_rated, on="candidate_item_id", how="left_anti")
    .groupBy("user_id", "candidate_item_id")
    .agg(
        F.sum(F.col("neighbor_rating") * F.col("similarity")).alias("score_numerator"),
        F.sum(F.abs("similarity")).alias("score_denominator"),
        F.count("*").alias("support")
    )
    .withColumn(
        "predicted_score",
        F.when(F.col("score_denominator") > 0,
               F.col("score_numerator") / F.col("score_denominator"))
         .otherwise(F.lit(0.0))
    )
    .join(item_lookup.withColumnRenamed("item_id", "candidate_item_id"), on="candidate_item_id", how="left")
    .withColumnRenamed("product_name", "candidate_item_name")
    .orderBy(F.desc("predicted_score"), F.desc("support"))
)

display(user_recommendations.limit(20))


In [0]:
def get_similar_users(user_id, top_n=10):
    sim_df = spark.table("workspace.default.sephora_user_user_similarity")
    result = (
        sim_df.filter(F.col("user_id") == str(user_id))
              .orderBy(F.desc("similarity"), F.desc("co_rating_count"))
              .limit(top_n)
    )
    return result

def recommend_for_user(user_id, top_n=10):
    ratings_used = ratings.select("user_id", "item_id", "rating")
    neighbors = spark.table("workspace.default.sephora_user_user_similarity")
    item_lookup = ratings.select("item_id", "product_name").dropDuplicates(["item_id"])

    user_history = ratings_used.filter(F.col("user_id") == str(user_id))
    already_rated = user_history.select(F.col("item_id").alias("candidate_item_id"))

    neighbor_history = (
        neighbors.alias("n")
        .join(
            ratings_used.alias("r"),
            F.col("n.neighbor_user_id") == F.col("r.user_id"),
            how="inner"
        )
        .select(
            F.col("n.user_id"),
            F.col("r.item_id").alias("candidate_item_id"),
            F.col("r.rating").alias("neighbor_rating"),
            F.col("n.similarity"),
            F.col("n.co_rating_count")
        )
    )

    recs = (
        neighbor_history
        .join(already_rated, on="candidate_item_id", how="left_anti")
        .groupBy("user_id", "candidate_item_id")
        .agg(
            F.sum(F.col("neighbor_rating") * F.col("similarity")).alias("score_numerator"),
            F.sum(F.abs("similarity")).alias("score_denominator"),
            F.count("*").alias("support")
        )
        .withColumn(
            "predicted_score",
            F.when(F.col("score_denominator") > 0,
                   F.col("score_numerator") / F.col("score_denominator"))
             .otherwise(F.lit(0.0))
        )
        .join(item_lookup.withColumnRenamed("item_id", "candidate_item_id"), on="candidate_item_id", how="left")
        .withColumnRenamed("product_name", "candidate_item_name")
        .orderBy(F.desc("predicted_score"), F.desc("support"))
        .limit(top_n)
    )
    return recs


In [0]:
# Example 1: similar users for a target user
display(get_similar_users("10004195528", top_n=10))

# Example 2: recommendations for a user
display(recommend_for_user("10004195528", top_n=10))


In [0]:
user_recommendations.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.user_recommendations_ubcf"
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Holdout scoring for UBCF using TRAIN-built neighbors
train_history = ratings.select("user_id", "item_id", "rating")
user_avg_test = user_avg.select("user_id", "user_avg_rating")
global_avg = ratings.agg(F.avg("rating").alias("global_avg_rating")).collect()[0][0]

neighbors = user_neighbors_topk.select(
    "user_id",
    "neighbor_user_id",
    "similarity"
)

# For each test pair (target user, target item), look up similar users who rated that item in TRAIN
candidate_contribs = (
    ratings_test.alias("t")
    .join(neighbors.alias("n"), F.col("t.user_id") == F.col("n.user_id"), how="left")
    .join(
        ratings_centered.alias("h"),
        (F.col("n.neighbor_user_id") == F.col("h.user_id")) &
        (F.col("t.item_id") == F.col("h.item_id")),
        how="left"
    )
    .filter(F.col("h.rating_centered").isNotNull())
    .select(
        F.col("t.user_id"),
        F.col("t.item_id"),
        F.col("t.product_name"),
        F.col("t.rating").alias("actual_rating"),
        F.col("n.similarity"),
        F.col("h.rating_centered"),
        (F.col("n.similarity") * F.col("h.rating_centered")).alias("weighted_centered")
    )
)

pred_centered = (
    candidate_contribs
    .groupBy("user_id", "item_id", "product_name", "actual_rating")
    .agg(
        F.sum("weighted_centered").alias("score_numerator"),
        F.sum(F.abs("similarity")).alias("score_denominator"),
        F.count("*").alias("support")
    )
    .withColumn(
        "pred_centered",
        F.when(F.col("score_denominator") > 0,
               F.col("score_numerator") / F.col("score_denominator"))
         .otherwise(F.lit(None))
    )
)

item_avg_fallback = ratings.groupBy("item_id").agg(F.avg("rating").alias("item_avg_rating"))

df_pred = (
    ratings_test.alias("t")
    .join(pred_centered.alias("p"), on=["user_id", "item_id", "product_name"], how="left")
    .join(user_avg_test, on="user_id", how="left")
    .join(item_avg_fallback, on="item_id", how="left")
    .withColumn(
        "prediction_raw",
        F.when(F.col("pred_centered").isNotNull(), F.col("user_avg_rating") + F.col("pred_centered"))
         .otherwise(F.coalesce(F.col("user_avg_rating"), F.col("item_avg_rating"), F.lit(global_avg)))
    )
    .withColumn(
        "prediction",
        F.when(F.col("prediction_raw") < 1.0, F.lit(1.0))
         .when(F.col("prediction_raw") > 5.0, F.lit(5.0))
         .otherwise(F.col("prediction_raw"))
    )
    .select(
        "user_id",
        "item_id",
        "product_name",
        F.col("t.rating").alias("actual_rating"),
        "prediction",
        F.coalesce(F.col("p.support"), F.lit(0)).alias("support")
    )
)

display(df_pred.limit(20))
print("Scored holdout rows:", df_pred.count())

# RMSE on holdout
rmse = (
    df_pred
    .select(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("actual_rating"), 2))).alias("rmse"))
    .collect()[0][0]
)

# MAE
mae = (
    df_pred
    .select(F.mean(F.abs(F.col("prediction") - F.col("actual_rating"))).alias("mae"))
    .collect()[0]["mae"]
)

# Ranking metrics from scored holdout pairs
K = 10
rank_window = Window.partitionBy("user_id").orderBy(F.desc("prediction"), F.desc("support"), F.asc("item_id"))

ranked_pred = (
    df_pred
    .withColumn("rank", F.row_number().over(rank_window))
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
)

topk = ranked_pred.filter(F.col("rank") <= K)

user_relevant_counts = (
    df_pred
    .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    .groupBy("user_id")
    .agg(F.sum("relevant").alias("n_relevant"))
)

per_user_hits = (
    topk.groupBy("user_id")
    .agg(F.sum("relevant").alias("hits"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn("precision_at_k", F.col("hits") / F.lit(K))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
    )
)

# MAP@K
ap_base = (
    topk
    .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn(
        "precision_at_rank",
        F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("precision_at_rank").alias("sum_precisions"))
    .join(user_relevant_counts, on="user_id", how="left")
    .fillna(0, subset=["n_relevant"])
    .withColumn(
        "ap_at_k",
        F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
         .otherwise(F.lit(0.0))
    )
)

precision_at_k = per_user_hits.agg(F.avg("precision_at_k")).collect()[0][0]
recall_at_k = per_user_hits.agg(F.avg("recall_at_k")).collect()[0][0]
map_at_k = ap_base.agg(F.avg("ap_at_k")).collect()[0][0]

coverage = (
    topk.select("item_id").distinct().count() /
    df_pred.select("item_id").distinct().count()
    if df_pred.select("item_id").distinct().count() > 0 else 0.0
)

# TRUE catalogue coverage (no min rating filters)
total_catalog_items = ratings.select("item_id").distinct().count()

catalogue_coverage = (
    topk.select("item_id").distinct().count() /
    total_catalog_items
    if total_catalog_items > 0 else 0.0
)

# NDCG@K (SAFE VERSION)
dcg_base = (
    topk
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("relevant") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("user_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg = (
    user_relevant_counts
    .withColumn(
        "ideal_dcg",
        F.expr(f"""
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        """)
    )
)

ndcg_df = (
    dcg_base
    .join(ideal_dcg, on="user_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k = (
    ndcg_df
    .select(F.avg("ndcg_at_k"))
    .collect()[0][0]
)

random_hit_rate = df_pred.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
lift_over_random = precision_at_k / random_hit_rate if random_hit_rate and random_hit_rate > 0 else None

# Inter-user diversity: 1 - average Jaccard overlap across top-K item lists
user_lists = topk.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
user_pairs = (
    user_lists.alias("a")
    .crossJoin(user_lists.alias("b"))
    .filter(F.col("a.user_id") < F.col("b.user_id"))
    .select(
        F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
        F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
    )
    .withColumn(
        "inter_user_diversity",
        F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
    )
)
inter_user_diversity = user_pairs.agg(F.avg("inter_user_diversity")).collect()[0][0]

# Serendipity proxy: relevant recommendations that are not among the most popular training items
item_popularity = ratings.groupBy("item_id").agg(F.count("*").alias("popularity"))
pop_threshold = item_popularity.approxQuantile("popularity", [0.8], 0.01)[0] if item_popularity.count() > 0 else 0

serendipity = (
    topk.join(item_popularity, on="item_id", how="left")
    .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
    .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
    .groupBy("user_id")
    .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
)
serendipity_score = serendipity.agg(F.avg("serendipity")).collect()[0][0]

metrics_df = spark.createDataFrame([
    ("RMSE", float(rmse) if rmse is not None else None),
    ("MAE", float(mae) if mae is not None else None),
    (f"Precision@{K}", float(precision_at_k)),
    (f"Recall@{K}", float(recall_at_k)),
    (f"MAP@{K}", float(map_at_k)),
    (f"NDCG@{K}", float(ndcg_at_k)),
    ("Catalogue coverage", float(catalogue_coverage)),
    ("Lift over random", float(lift_over_random)),
    ("Serendipity", float(serendipity_score)),
    ("Inter-user diversity", float(inter_user_diversity)),
], ["metric", "value"])

display(metrics_df)
